In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import sys
sys.path.insert(0, '/content/drive/MyDrive/heart-murmur')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import shutil, os

# Copy spectrograms vào /content/
print('Copying spectrograms...')
shutil.copytree(
    '/content/drive/MyDrive/heart-murmur/data/processed/spectrograms',
    '/content/spectrograms'
)

print('Copying models...')
shutil.copytree(
    '/content/drive/MyDrive/heart-murmur/models/rnn',
    '/content/models_rnn'
)

print('Copying metadata...')
os.makedirs('/content/metadata', exist_ok=True)
for f in ['cv_splits.json', 'recordings.csv', 'patients.csv']:
    shutil.copy(
        f'/content/drive/MyDrive/heart-murmur/data/metadata/{f}',
        f'/content/metadata/{f}'
    )

# Copy recording_results
shutil.copy(
    '/content/drive/MyDrive/heart-murmur/experiments/results/recording_results.csv',
    '/content/recording_results.csv'
)

print('Done copying.')

Copying spectrograms...
Copying models...
Copying metadata...
Done copying.


In [ ]:
import numpy as np
import pandas as pd
import torch
import json
import time
from pathlib import Path

SPEC_DIR  = Path('/content/spectrograms')
MODEL_DIR = Path('/content/models_rnn')

from src.models.rnn import build_model
from src.models.parallel_hsmm import run_parallel_hsmm

In [ ]:
models = {}
for fold_name in ['fold_0', 'fold_1', 'fold_2', 'fold_3', 'fold_4']:
    ckpt = torch.load(MODEL_DIR / f'{fold_name}_best.pt', map_location='cpu')
    m    = build_model(seed=42)
    m.load_state_dict(ckpt['model_state_dict'])
    m.eval()
    models[fold_name] = m
    print(f'Loaded {fold_name}')

Loaded fold_0
Loaded fold_1
Loaded fold_2
Loaded fold_3
Loaded fold_4


In [ ]:
df_pat = pd.read_csv('/content/metadata/patients.csv')
df_rec = pd.read_csv('/content/recording_results.csv')

with open('/content/metadata/cv_splits.json') as f:
    cv_splits = json.load(f)

# Mapping patient_id → true_murmur
pat_murmur = dict(zip(df_pat['patient_id'], df_pat['murmur']))

SUBSET_PER_FOLD = 10
np.random.seed(42)

val_present_recs = []
for fold_name, fold_data in cv_splits.items():
    present_recs = []
    for rec_id in fold_data['val_recordings']:
        pat_id = int(rec_id.split('_')[0])
        if pat_murmur.get(pat_id) == 'Present':
            present_recs.append(rec_id)
    if len(present_recs) > SUBSET_PER_FOLD:
        chosen = list(np.random.choice(present_recs, SUBSET_PER_FOLD, replace=False))
    else:
        chosen = present_recs
    for rec_id in chosen:
        val_present_recs.append((rec_id, fold_name))

print(f'Total: {len(val_present_recs)} recordings')

# Baseline C(M−N)
baseline_cmn = {}
for rec_id, _ in val_present_recs:
    row = df_rec[df_rec['recording_id'] == rec_id]
    if len(row) > 0:
        baseline_cmn[rec_id] = row['c_mn'].values[0]

print(f'Baseline loaded: {len(baseline_cmn)} recordings')
print(f'Mean baseline C(M−N): {np.mean(list(baseline_cmn.values())):.4f}')

Total: 50 recordings
Baseline loaded: 50 recordings
Mean baseline C(M−N): 0.1243


In [ ]:
N_BINS    = 41
n_recs    = len(val_present_recs)
delta_cmn = np.zeros((N_BINS, n_recs))

print(f'Running ablation: {N_BINS} bins × {n_recs} recordings on GPU/CPU...')
t_start = time.time()

for b in range(N_BINS):
    for r_idx, (rec_id, fold_name) in enumerate(val_present_recs):
        spec           = np.load(SPEC_DIR / f'{rec_id}.npy')
        spec_ablated   = spec.copy()
        spec_ablated[b, :] = 0.0

        model = models[fold_name]
        T     = spec_ablated.shape[1]
        with torch.no_grad():
            x      = torch.FloatTensor(spec_ablated.T).unsqueeze(0)
            logits = model(x, [T])
            post   = torch.softmax(logits, dim=-1).squeeze(0).numpy()

        hsmm_out             = run_parallel_hsmm(post, feature_rate=50)
        c_mn_abl             = hsmm_out['murmur_conf'] - hsmm_out['healthy_conf']
        baseline             = baseline_cmn.get(rec_id, 0.0)
        delta_cmn[b, r_idx] = baseline - c_mn_abl

    elapsed = time.time() - t_start
    eta     = elapsed / (b + 1) * (N_BINS - b - 1)
    print(f'Bin {b:2d} ({b*20:4d} Hz) done | '
          f'importance={delta_cmn[b].mean():.4f} | '
          f'elapsed={elapsed/60:.1f}min ETA={eta/60:.1f}min')

importance = delta_cmn.mean(axis=1)
print(f'\nDone! Total: {(time.time()-t_start)/60:.1f} min')
print(f'Peak bin: {importance.argmax()} ({importance.argmax()*20} Hz)')

Running ablation: 41 bins × 50 recordings on GPU/CPU...
Bin  0 (   0 Hz) done | importance=0.0044 | elapsed=2.2min ETA=86.0min
Bin  1 (  20 Hz) done | importance=0.0031 | elapsed=4.3min ETA=83.6min
Bin  2 (  40 Hz) done | importance=-0.0032 | elapsed=6.5min ETA=82.3min
Bin  3 (  60 Hz) done | importance=0.0031 | elapsed=8.6min ETA=79.9min
Bin  4 (  80 Hz) done | importance=0.0004 | elapsed=10.8min ETA=77.7min
Bin  5 ( 100 Hz) done | importance=0.0040 | elapsed=12.9min ETA=75.5min
Bin  6 ( 120 Hz) done | importance=0.0103 | elapsed=15.1min ETA=73.3min
Bin  7 ( 140 Hz) done | importance=0.0114 | elapsed=17.2min ETA=71.1min
Bin  8 ( 160 Hz) done | importance=0.0109 | elapsed=19.4min ETA=68.9min
Bin  9 ( 180 Hz) done | importance=0.0086 | elapsed=21.5min ETA=66.8min
Bin 10 ( 200 Hz) done | importance=0.0052 | elapsed=23.7min ETA=64.6min
Bin 11 ( 220 Hz) done | importance=0.0013 | elapsed=25.8min ETA=62.5min
Bin 12 ( 240 Hz) done | importance=0.0025 | elapsed=28.0min ETA=60.3min
Bin 13 ( 26

In [ ]:
import os
os.makedirs('/content/drive/MyDrive/heart-murmur/experiments/results', exist_ok=True)

np.save('/content/drive/MyDrive/heart-murmur/experiments/results/ablation_importance.npy', importance)
np.save('/content/drive/MyDrive/heart-murmur/experiments/results/ablation_delta_cmn.npy', delta_cmn)
print('Saved to Drive.')
print(f'importance shape: {importance.shape}')
print(f'delta_cmn shape:  {delta_cmn.shape}')

Saved to Drive.
importance shape: (41,)
delta_cmn shape:  (41, 50)
